## EXAMPLE INTRO

Έστω ένα ελάχιστο σύστημα ισχύος που αποτελείται από δύο διαύλους συνδεδεμένους με μια γραμμή μεταφοράς. Μια γεννήτρια τοποθετείται στο ένα άκρο και ένα φορτίο στο άλλο. Αυτή η απλή διάταξη χρησιμεύει για να δείξει πώς υπολογίζονται οι ροές ισχύος και πώς προσδιορίζονται τα προφίλ τάσης χρησιμοποιώντας το network.pf().

<div class="container">
  <img src="/imgs/example_intro.png" alt="Περιγραφή" style=" display: block;
  margin-left: auto;
  margin-right: auto;
  width: 50%;">
</div>

> *Το διάγραμμα δείχνει ένα σύστημα ισχύος με τη γεννήτρια Α συνδεδεμένη σε έναν δίαυλο με την ένδειξη 220 κιλοβόλτ. Αυτός ο δίαυλος συνδέεται με έναν άλλο δίαυλο, επίσης με την ένδειξη 220 κιλοβόλτ, μέσω της γραμμής 1. Η γραμμή 1 έχει παραμέτρους x ίση με 50 χιλιόμετρα, r ίση με 0,1 και s δείκτη nom ίση με 100. Ο δεύτερος δίαυλος συνδέεται με το φορτίο Α, το οποίο έχει παράμετρο p δείκτη set ίση με 70.* 





In [ ]:
import pypsa
import numpy as np

#Νέο Δίκτυο (Δήλωση)
net = pypsa.Network()

#Προσθέτουμε 2 Ζυγούς υψηλής τάσης 
net.add("Bus", "Bus A", v_nom=220)
net.add("Bus", "Bus B", v_nom=220)

#Συνδέουμε τους Ζυγούς μέσω γραμμής μεταφοράς
net.add("Line", "Line 1",
        bus0 = "Bus A",
        bus1 = "Bus B",
        length = 50,
        x=0.1,
        r=0.01,
        s_nom = 100)

#Προσθέτουμε μια γεννήτρια (Η/Ζ) στο ΖΥΓΟ Α
net.add("Generator", "Generator A",
        bus="Bus A",
        p_nom = 80,
        marginal_cost=50)

#Προσθέτουμε ένα φορτίο στο ΖΥΓΟ Β
net.add("Load", "Load A",
        bus="Bus B",
        p_set = 70)

#Τρέξε βασική ανάλυση ροής ρεύματος
net.pf()

net.pf()

print("\n" + "="*30)
print(" ΑΝΑΦΟΡΑ ΡΟΗΣ ΙΣΧΥΟΣ (AC) ")
print("="*30)

# 1. Αποτελέσματα Ζυγών (Τάσεις και Γωνίες)
print("\n[ ΚΑΤΑΣΤΑΣΗ ΖΥΓΩΝ ]")
for bus_name in net.buses.index:
    v_mag = net.buses_t.v_mag_pu.at["now", bus_name]
    v_ang = np.rad2deg(net.buses_t.v_ang.at["now", bus_name])
    print(f"Ζυγός {bus_name}: Τάση = {v_mag:.4f} p.u. | Γωνία = {v_ang:.2f}°")

# 2. Αποτελέσματα Γραμμών (Ροές και Φόρτιση)
print("\n[ ΡΟΕΣ ΓΡΑΜΜΩΝ ]")
for line_name in net.lines.index:
    p0 = net.lines_t.p0.at["now", line_name]
    s_nom = net.lines.at[line_name, "s_nom"]
    loading = (abs(p0) / s_nom) * 100
    print(f"Γραμμή {line_name}: Ροή = {p0:.2f} MW | Φόρτιση = {loading:.1f}%")

# 3. Πραγματική Παραγωγή (Dispatch)
print("\n[ ΠΑΡΑΓΩΓΗ ΓΕΝΝΗΤΡΙΩΝ ]")
# Υπολογίζουμε την παραγωγή από τις εγχύσεις στους ζυγούς για να αποφύγουμε το NaN
for gen_name in net.generators.index:
    bus_id = net.generators.at[gen_name, "bus"]
    # Η έγχυση στον ζυγό ισούται με την παραγωγή (αν δεν υπάρχει άλλο στοιχείο εκεί)
    actual_p = net.buses_t.p.at["now", bus_id] 
    print(f"Γεννήτρια {gen_name}: Ισχύς Εξόδου = {actual_p:.4f} MW")

# 4. Υπολογισμός Απωλειών
total_losses = net.lines_t.p0.sum().sum() - abs(net.lines_t.p1.sum().sum())
print(f"\nΣυνολικές Απώλειες Δικτύου: {abs(total_losses)*1000:.2f} kW")
print("="*30)


INFO:pypsa.network.power_flow:Performing non-linear load-flow on AC sub-network <pypsa.networks.SubNetwork object at 0x000001937B882570> for snapshots Index(['now'], dtype='str', name='snapshot')
INFO:pypsa.network.power_flow:Performing non-linear load-flow on AC sub-network <pypsa.networks.SubNetwork object at 0x000001937B882990> for snapshots Index(['now'], dtype='str', name='snapshot')



 ΑΝΑΦΟΡΑ ΡΟΗΣ ΙΣΧΥΟΣ (AC) 

[ ΚΑΤΑΣΤΑΣΗ ΖΥΓΩΝ ]
Ζυγός Bus A: Τάση = 1.0000 p.u. | Γωνία = 0.00°
Ζυγός Bus B: Τάση = 1.0000 p.u. | Γωνία = -0.01°

[ ΡΟΕΣ ΓΡΑΜΜΩΝ ]
Γραμμή Line 1: Ροή = 70.00 MW | Φόρτιση = 70.0%

[ ΠΑΡΑΓΩΓΗ ΓΕΝΝΗΤΡΙΩΝ ]
Γεννήτρια Generator A: Ισχύς Εξόδου = 70.0010 MW

Συνολικές Απώλειες Δικτύου: 1.01 kW


1. Κατάσταση Ζυγών (Τάση και Γωνία)

    Bus A (1.0000 p.u. | 0.00°): Είναι ο Slack Bus (ζυγός αναφοράς). Η τάση είναι τέλεια (100%) και η γωνία είναι το σημείο μηδέν από όπου ξεκινά η μέτρηση για όλο το δίκτυο.

    Bus B (1.0000 p.u. | -0.01°): * Η τάση παραμένει στο 1.0000 (ή πολύ κοντά σε αυτό), πράγμα που σημαίνει ότι δεν έχεις πρόβλημα πτώσης τάσης.

        Η αρνητική γωνία (-0.01°) είναι το "κλειδί". Επιβεβαιώνει ότι η ισχύς ρέει από το Α προς το Β. Η διαφορά είναι πολύ μικρή, που σημαίνει ότι η γραμμή μεταφέρει τα 70 MW χωρίς να "κουράζεται" ηλεκτρικά.

2. Ροές Γραμμών και Φόρτιση

    Ροή 70.00 MW: Η γραμμή μεταφέρει ακριβώς όση ισχύ ζητάει το φορτίο στον Ζυγό Β.

    Φόρτιση 70.0%: Επειδή το όριο της γραμμής (s_nom) είναι 100 MW, βρίσκεσαι στο 70%.

        Συμπέρασμα: Το δίκτυο είναι ασφαλές, αλλά αν το φορτίο αυξηθεί πάνω από τα 100 MW, η γραμμή θα υπερφορτωθεί (συμφόρηση).

3. Παραγωγή Γεννητριών (Η λύση στο NaN)

    Ισχύς Εξόδου 70.0010 MW: * Εδώ βλέπεις την πραγματική δουλειά της γεννήτριας.

        Παράγει τα 70 MW για το φορτίο + 0.0010 MW (δηλαδή 1 kW) για τις απώλειες της γραμμής.

        Αυτό το επιπλέον ποσό είναι που έκανε το PyPSA να σου βγάζει NaN πριν, γιατί δεν μπορούσε να το ξέρει μέχρι να λύσει τις εξισώσεις.

4. Συνολικές Απώλειες Δικτύου (1.01 kW)

    Αντιπροσωπεύουν την ενέργεια που χάνεται ως θερμότητα πάνω στα καλώδια της γραμμής λόγω της αντίστασης (r=0.01).

    Σχόλιο: Οι απώλειες είναι πολύ χαμηλές (0.0014% της συνολικής ισχύος), γεγονός που υποδηλώνει πολύ αποδοτική μεταφορά σε αυτή την τάση (220 kV).



    >Η ανάλυση ροής ισχύος AC κατέδειξε ένα ηλεκτρικά εύρωστο σύστημα. Η γεννήτρια αναφοράς (Generator A) καλύπτει επιτυχώς το φορτίο των 70 MW καθώς και τις ωμικές απώλειες της τάξης του 1.01 kW. Η φόρτιση της γραμμής διασύνδεσης ανέρχεται στο 70%, διατηρώντας επαρκές περιθώριο ασφαλείας, ενώ η απειροελάχιστη πτώση τάσης και η μικρή γωνιακή απόκλιση (-0.01°) υποδηλώνουν υψηλή στατική ευστάθεια στο εξεταζόμενο σενάριο.

In [15]:
import pypsa
import numpy as np

#Νέο Δίκτυο (Δήλωση)
net = pypsa.Network()

#Προσθέτουμε 2 Ζυγούς υψηλής τάσης 
net.add("Bus", "Bus A", v_nom=220)
net.add("Bus", "Bus B", v_nom=220)

#Συνδέουμε τους Ζυγούς μέσω γραμμής μεταφοράς
net.add("Line", "Line 1",
        bus0 = "Bus A",
        bus1 = "Bus B",
        length = 50,
        x=0.1,
        r=0.01,
        s_nom = 100)

#Προσθέτουμε μια γεννήτρια (Η/Ζ) στο ΖΥΓΟ Α
net.add("Generator", "Generator A",
        bus="Bus A",
        p_nom = 80,
        marginal_cost=50)

#Προσθέτουμε ένα φορτίο στο ΖΥΓΟ Β
net.add("Load", "Load A",
        bus="Bus B",
        p_set = 100)

#Τρέξε βασική ανάλυση ροής ρεύματος
net.pf()

net.pf()

print("\n" + "="*30)
print(" ΑΝΑΦΟΡΑ ΡΟΗΣ ΙΣΧΥΟΣ (AC) ")
print("="*30)

# 1. Αποτελέσματα Ζυγών (Τάσεις και Γωνίες)
print("\n[ ΚΑΤΑΣΤΑΣΗ ΖΥΓΩΝ ]")
for bus_name in net.buses.index:
    v_mag = net.buses_t.v_mag_pu.at["now", bus_name]
    v_ang = np.rad2deg(net.buses_t.v_ang.at["now", bus_name])
    print(f"Ζυγός {bus_name}: Τάση = {v_mag:.4f} p.u. | Γωνία = {v_ang:.2f}°")

# 2. Αποτελέσματα Γραμμών (Ροές και Φόρτιση)
print("\n[ ΡΟΕΣ ΓΡΑΜΜΩΝ ]")
for line_name in net.lines.index:
    p0 = net.lines_t.p0.at["now", line_name]
    s_nom = net.lines.at[line_name, "s_nom"]
    loading = (abs(p0) / s_nom) * 100
    print(f"Γραμμή {line_name}: Ροή = {p0:.2f} MW | Φόρτιση = {loading:.1f}%")

# 3. Πραγματική Παραγωγή (Dispatch)
print("\n[ ΠΑΡΑΓΩΓΗ ΓΕΝΝΗΤΡΙΩΝ ]")
# Υπολογίζουμε την παραγωγή από τις εγχύσεις στους ζυγούς για να αποφύγουμε το NaN
for gen_name in net.generators.index:
    bus_id = net.generators.at[gen_name, "bus"]
    # Η έγχυση στον ζυγό ισούται με την παραγωγή (αν δεν υπάρχει άλλο στοιχείο εκεί)
    actual_p = net.buses_t.p.at["now", bus_id] 
    print(f"Γεννήτρια {gen_name}: Ισχύς Εξόδου = {actual_p:.4f} MW")

# 4. Υπολογισμός Απωλειών
total_losses = net.lines_t.p0.sum().sum() - abs(net.lines_t.p1.sum().sum())
print(f"\nΣυνολικές Απώλειες Δικτύου: {abs(total_losses)*1000:.2f} kW")
print("="*30)







INFO:pypsa.network.power_flow:Performing non-linear load-flow on AC sub-network <pypsa.networks.SubNetwork object at 0x000001937B8824C0> for snapshots Index(['now'], dtype='str', name='snapshot')
INFO:pypsa.network.power_flow:Performing non-linear load-flow on AC sub-network <pypsa.networks.SubNetwork object at 0x000001937B882620> for snapshots Index(['now'], dtype='str', name='snapshot')



 ΑΝΑΦΟΡΑ ΡΟΗΣ ΙΣΧΥΟΣ (AC) 

[ ΚΑΤΑΣΤΑΣΗ ΖΥΓΩΝ ]
Ζυγός Bus A: Τάση = 1.0000 p.u. | Γωνία = 0.00°
Ζυγός Bus B: Τάση = 1.0000 p.u. | Γωνία = -0.01°

[ ΡΟΕΣ ΓΡΑΜΜΩΝ ]
Γραμμή Line 1: Ροή = 100.00 MW | Φόρτιση = 100.0%

[ ΠΑΡΑΓΩΓΗ ΓΕΝΝΗΤΡΙΩΝ ]
Γεννήτρια Generator A: Ισχύς Εξόδου = 100.0021 MW

Συνολικές Απώλειες Δικτύου: 2.07 kW


1. Ροές Γραμμών: Το "Κρίσιμο" Σημείο

    Φόρτιση 100.0%: Αυτό είναι το πιο σημαντικό εύρημα. Η γραμμή μεταφέρει 100 MW, που είναι η ονομαστική της χωρητικότητα (s_nom = 100).

    Ερμηνεία: Το δίκτυο βρίσκεται σε κατάσταση συμφόρησης (congestion). Οποιαδήποτε περαιτέρω αύξηση στη ζήτηση του Ζυγού Β θα προκαλέσει υπερφόρτωση της γραμμής, κάτι που σε πραγματικές συνθήκες θα ενεργοποιούσε τα συστήματα προστασίας για την αποφυγή υπερθέρμανσης των καλωδίων.

2. Παραγωγή και Απώλειες: Η Μη Γραμμική Αύξηση

    Ισχύς Εξόδου 100.0021 MW: Η γεννήτρια παράγει πλέον 100 MW για το φορτίο και επιπλέον 2.1 Watt για τις απώλειες.

    Απώλειες Δικτύου 2.07 kW: Παρατήρησε κάτι ενδιαφέρον: Όταν το φορτίο αυξήθηκε από τα 70 στα 100 MW (αύξηση ~43%), οι απώλειες αυξήθηκαν από το 1.01 kW στα 2.07 kW (διπλασιάστηκαν).

    Η Φυσική πίσω από αυτό: Οι απώλειες ισχύος στις γραμμές ακολουθούν τον νόμο Ploss​=I2⋅R. Δηλαδή, οι απώλειες αυξάνονται με το τετράγωνο του ρεύματος. Αυτός είναι ο λόγος που η διαχείριση της φόρτισης είναι κρίσιμη για την αποδοτικότητα.

3. Κατάσταση Ζυγών: Ευστάθεια υπό Πίεση

    Γωνία -0.01°: Παρόλο που η γραμμή είναι στο 100% της φόρτισης, η γωνιακή απόκλιση παραμένει πολύ μικρή. Αυτό οφείλεται στο ότι η γραμμή σου έχει πολύ χαμηλή επαγωγική αντίδραση (x=0.1).

    Τάση 1.0000 p.u.: Η τάση στον Ζυγό Β παραμένει σταθερή. Σε ένα πραγματικό σύστημα AC, αν η γραμμή ήταν πολύ μακριά, η τάση θα άρχιζε να "βυθίζεται" (voltage drop) καθώς πλησιάζουμε το 100% της φόρτισης. Εδώ, η μικρή απόσταση (50χλμ) βοηθά στη διατήρηση του προφίλ τάσης.


    >Στη συγκεκριμένη προσομοίωση, το σύστημα αγγίζει τα λειτουργικά του όρια, με τη γραμμή διασύνδεσης να εμφανίζει 100% φόρτιση. Η ανάλυση αναδεικνύει τη μη γραμμική φύση των απωλειών μεταφοράς, οι οποίες διπλασιάστηκαν με την αύξηση του φορτίου, επιβεβαιώνοντας τη θερμική καταπόνηση του στοιχείου. Παρόλο που το προφίλ τάσης παραμένει εντός προδιαγραφών (1.0 p.u.), το σύστημα στερείται πλέον περιθωρίου ασφαλείας (security margin), καθιστώντας αναγκαία είτε την ανακατανομή της παραγωγής (redispatch) είτε την ενίσχυση της υποδομής για την κάλυψη μελλοντικών φορτίων.